In [3]:
pip install groq

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp311-cp311-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp311-cp311-win_amd64.whl (2.0 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   ----- ---------------------------------- 1/7 [sniffio]
   ----------- ---------------------------- 2/7 [pydantic-core]
   ----------------- ---------------------- 3/7 [distro]
   ---------------------------- ----------- 5/7 [pydantic]
   ---------------------------- ----------- 5/7 [pydantic]
   ---------------------------- ----------- 5/7 [pydantic]
   ---------------------------- ----------- 5/7 [pydantic]
   ---------------------------- ----------- 5/7 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ================================
# IMPORTS
# ================================
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq


# ================================
# CONFIG
# ================================
# Load key from text file
with open("E:\Krishisamadhan\API.txt", "r") as file:
    groq_api_key = file.read().strip()

# Initialize your Groq client or environment
import os
os.environ["GROQ_API_KEY"] = groq_api_key
faiss_index_path = "data/vector_store/faiss_index.bin"
metadata_path = "data/vector_store/metadata.json"


# ================================
# LOAD DATA
# ================================
index = faiss.read_index(faiss_index_path)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = Groq(api_key=GROQ_API_KEY)


# ================================
# QUERY PROCESSING
# ================================
def process_query(query):
    expanded = query + " fertilizer soil nutrients crop farming agriculture"
    return "Represent this sentence for searching relevant passages: " + expanded


# ================================
# RETRIEVAL (IMPROVED)
# ================================
def retrieve(query, top_k=8):
    query = process_query(query)

    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k * 3)

    results = []
    seen = set()

    for i, idx in enumerate(indices[0]):
        item = metadata[idx]
        score = float(distances[0][i])

        if item["text"] in seen:
            continue
        seen.add(item["text"])

        results.append({
            "text": item["text"],
            "source": item["source"],
            "domain": item["domain"],
            "score": score
        })

        if len(results) >= top_k:
            break

    return results


# ================================
# CONTEXT BUILDER (BETTER FORMAT)
# ================================
def build_context(chunks):
    context = ""

    for i, chunk in enumerate(chunks):
        context += f"\n[Chunk {i+1} | Domain: {chunk['domain']}]\n"
        context += chunk["text"][:500] + "\n"

    return context


# ================================
# FALLBACK CHECK 🔥
# ================================
def is_low_confidence(chunks, threshold=0.4):
    scores = [c["score"] for c in chunks]
    avg_score = np.mean(scores) if scores else 0
    return avg_score < threshold


# ================================
# GENERATE ANSWER (SMART)
# ================================
def generate_answer(query):

    chunks = retrieve(query)

    context = build_context(chunks)

    low_conf = is_low_confidence(chunks)

    # 🔥 Improved Prompt
    prompt = f"""
You are KrishiSamadhan, an expert agricultural advisor.

Instructions:
- Give clear, step-by-step farming advice
- Mention fertilizer type, quantity, and timing
- Be practical and easy to understand
- If exact info is missing, give closest useful advice

Context:
{context}

Question:
{query}

Answer:
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=500
        )
        

        answer = response.choices[0].message.content

    except Exception as e:
        print("Error:", e)
        answer = "⚠️ Model is temporarily unavailable. Please try again."

    # 🔥 Add fallback message if low confidence
    if low_conf:
        answer = (
            "⚠️ I don't have very specific information for this query, "
            "but based on related agricultural knowledge:\n\n"
            + answer
        )

    sources = list(set([c["source"] for c in chunks]))

    return answer, sources


# ================================
# TEST
# ================================
query = "What fertilizer is best for wheat crop?"

print("\n🔹 Query:\n", query)

answer, sources = generate_answer(query)

print("\n🔹 Answer:\n")
print(answer)

print("\n🔹 Sources:\n")
for s in sources:
    print("-", s)

e:\Udemy ML course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2948.50it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔹 Query:
 What fertilizer is best for wheat crop?

🔹 Answer:

As an agricultural advisor, I'd recommend a balanced fertilizer for wheat crops. Considering the nutrient requirements of wheat, I would suggest a fertilizer that contains Nitrogen (N), Phosphorus (P), and Potassium (K) in the following proportions:

- Nitrogen (N): 10-15% (to promote leaf growth and grain development)
- Phosphorus (P): 10-15% (to support root development and grain formation)
- Potassium (K): 10-15% (to enhance overall plant health and resistance to disease)

In terms of specific fertilizer types, you can consider the following options:

1. **DAP (Diammonium Phosphate)**: A popular choice for wheat crops, DAP provides a balanced mix of N and P.
2. **Urea**: A good source of N, urea can be applied in combination with other fertilizers to meet the P and K requirements.
3. **NPK (Nitrogen-Phosphorus-Potassium) Fertilizer**: A balanced fertilizer that contains N, P, and K in the desired proportions.

When apply